# Part 3 — LoRA fine-tuning of Gemma 3 (1B) on TPU vs GPU

COSC 6385 — Experiment A, Part 3. Based on Google's *Fine-tune Gemma in Keras using LoRA* tutorial,
adapted to the **`gemma3_1b`** KerasHub preset (Keras 3 + JAX backend).

**Run this notebook unchanged on each accelerator** (`Runtime -> Change runtime type`):
at minimum **v5e-1 TPU** and **T4 GPU**. It records, per run: time per training epoch,
peak memory (if available), and sample generations **before and after** fine-tuning.
Download `gemma_lora_results.csv` after every run.

In [ ]:
# --- COSC6385 environment fix (Colab Nov 2026 image) ---
# Pin a flax compatible with the image's jax WITHOUT touching jax/jaxlib (protects libtpu),
# and re-add a jax attribute some libs still call. Harmless if unused by keras_hub.
import os, subprocess, sys
# Force kagglehub to download Gemma directly from kaggle.com instead of the Colab mount
# proxy, which 403s because it does not carry our KAGGLE_API_TOKEN (see DISABLE_COLAB_CACHE).
os.environ['DISABLE_COLAB_CACHE'] = 'true'
subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','flax==0.10.6'], check=False)
try:
    import jax
    from jax.extend.core import get_opaque_trace_state as _gots
    jax.core.get_opaque_trace_state = _gots
except Exception as e:
    print('shim skipped:', e)
print('env fix applied')

## 0. Setup
Needs a Kaggle account that has **accepted the Gemma 3 license**, with `KAGGLE_USERNAME`
and `KAGGLE_KEY` stored in Colab **Secrets** (key icon, left sidebar), toggled on for this notebook.

In [ ]:
# Pull Kaggle creds from Colab Secrets (key icon, left sidebar) into env.
# Add ONE of these as a Colab Secret (toggled on for this notebook):
#   KAGGLE_API_TOKEN = KGAT_...   (new single-token style), OR
#   KAGGLE_USERNAME + KAGGLE_KEY  (classic pair)
import os
try:
    from google.colab import userdata
    def _get(k):
        try: return userdata.get(k)
        except Exception: return None
    tok=_get('KAGGLE_API_TOKEN')
    if tok:
        os.environ['KAGGLE_API_TOKEN']=tok
        print('Kaggle: using KAGGLE_API_TOKEN from Secrets.')
    else:
        u,k=_get('KAGGLE_USERNAME'),_get('KAGGLE_KEY')
        if u and k:
            os.environ['KAGGLE_USERNAME']=u; os.environ['KAGGLE_KEY']=k
            print('Kaggle: using KAGGLE_USERNAME/KAGGLE_KEY from Secrets.')
        else:
            print('WARNING: no Kaggle secret found; Part 3 model download will fail.')
except Exception as e:
    print('Not on Colab or secrets missing:', type(e).__name__, e)


In [ ]:
# Keras 3 on the JAX backend. Do NOT `pip install -U jax` on a TPU runtime.
os.environ['KERAS_BACKEND'] = 'jax'
# Keep the JAX/XLA allocator from grabbing all HBM so we can read a peak-memory number.
os.environ.setdefault('XLA_PYTHON_CLIENT_MEM_FRACTION', '0.9')
import subprocess, sys, importlib
for pkg in ('keras-hub', 'keras'):
    try:
        importlib.import_module(pkg.replace('-', '_'))
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', pkg], check=True)

In [ ]:
import time, datetime, platform, json
import numpy as np, pandas as pd
import jax, keras, keras_hub

dev = jax.devices()[0]
PLATFORM, DEVICE_KIND = dev.platform, dev.device_kind
RUN_ID = f"{DEVICE_KIND.replace(' ', '_')}_{datetime.datetime.now():%Y%m%d_%H%M%S}"
ENV = dict(run_id=RUN_ID, platform=PLATFORM, device_kind=DEVICE_KIND,
           n_devices=jax.device_count(), jax=jax.__version__,
           keras=keras.__version__, keras_hub=keras_hub.__version__,
           python=platform.python_version(),
           colab_runtime=os.environ.get('COLAB_RELEASE_TAG', 'n/a'))
for k, v in ENV.items():
    print(f'{k:>14}: {v}')

## 1. Load Gemma 3 1B
The `gemma3_1b` preset is the 1B text model. If HBM is tight on the T4 we reduce the
sequence length (same value on both devices, so the comparison stays fair).

In [ ]:
SEQLEN = 256   # keep identical across devices; lower both if the T4 OOMs
EPOCHS = 3
BATCH  = 1
LORA_RANK = 4

t0 = time.perf_counter()
gemma = keras_hub.models.Gemma3CausalLM.from_preset('gemma3_1b')
gemma.preprocessor.sequence_length = SEQLEN
load_s = time.perf_counter() - t0
print(f'loaded gemma3_1b in {load_s:.1f}s')
gemma.summary()

## 2. A small instruction dataset
We fine-tune on a few hundred Databricks-Dolly-style instruction/response pairs,
formatted as a simple instruction template (same as the Google LoRA tutorial).

In [ ]:
# Small, self-contained instruction set so the notebook is reproducible offline too.
import urllib.request
PROMPT_TPL = 'Instruction:\n{instruction}\n\nResponse:\n'
prompts, responses = [], []
try:
    url = 'https://huggingface.co/datasets/databricks/databricks-dolly-15k/resolve/main/databricks-dolly-15k.jsonl'
    raw = urllib.request.urlopen(url, timeout=30).read().decode()
    for line in raw.splitlines():
        r = json.loads(line)
        if r.get('context'):
            continue  # keep only context-free examples
        prompts.append(PROMPT_TPL.format(instruction=r['instruction']))
        responses.append(r['response'])
        if len(prompts) >= 400:
            break
    src = 'dolly-15k (first 400 context-free)'
except Exception as e:
    print('download failed, using tiny built-in set:', type(e).__name__, e)
    pairs = [('What is the capital of France?', 'The capital of France is Paris.'),
             ('Name three primary colors.', 'Red, blue, and yellow.'),
             ('What is 2+2?', 'It is 4.')] * 40
    prompts = [PROMPT_TPL.format(instruction=i) for i, _ in pairs]
    responses = [r for _, r in pairs]
    src = 'built-in tiny set'
# Gemma 3's Gemma3CausalLMPreprocessor expects a DICT of prompts/responses
# (it concatenates them and masks the prompt tokens out of the loss),
# unlike Gemma 1/2 which accepted a flat list of strings.
data = {'prompts': prompts, 'responses': responses}
print(len(prompts), 'examples from', src)

## 3. Sample outputs BEFORE fine-tuning

In [ ]:
PROMPTS = [
    'Instruction:\nExplain what a TPU is in one sentence.\n\nResponse:\n',
    'Instruction:\nWhat is speculative execution?\n\nResponse:\n',
    'Instruction:\nGive me a haiku about GPUs.\n\nResponse:\n',
]
def sample(model, tag):
    outs = []
    for p in PROMPTS:
        o = model.generate(p, max_length=96)
        outs.append(o)
        print(f'--- [{tag}] {p!r}\n{o}\n')
    return outs
before = sample(gemma, 'before')

## 4. Enable LoRA and fine-tune
LoRA keeps the base weights frozen and trains only small rank-`r` adapters, so this fits
on a single free accelerator. We time **each epoch** and read peak memory afterwards.

In [ ]:
try:
    gemma.backbone.enable_lora(rank=LORA_RANK)
except ValueError:
    pass  # LoRA already enabled (e.g. re-running the cell)
gemma.summary()  # note the big drop in *trainable* params

gemma.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=keras.optimizers.Adam(learning_rate=5e-5),
    weighted_metrics=[keras.metrics.SparseCategoricalAccuracy()],
)

class EpochTimer(keras.callbacks.Callback):
    def on_epoch_begin(self, epoch, logs=None): self.t = time.perf_counter()
    def on_epoch_end(self, epoch, logs=None):
        dt = time.perf_counter() - self.t
        self.times = getattr(self, 'times', []) + [dt]
        print(f'  epoch {epoch}: {dt:.1f}s  loss={logs.get("loss"):.4f}')

timer = EpochTimer()
t0 = time.perf_counter()
gemma.fit(data, epochs=EPOCHS, batch_size=BATCH, callbacks=[timer])
train_s = time.perf_counter() - t0

In [ ]:
# Peak memory (best-effort; JAX exposes per-device stats on GPU/TPU).
peak_mb = None
try:
    st = jax.devices()[0].memory_stats()
    peak_mb = st.get('peak_bytes_in_use', st.get('bytes_in_use', 0)) / 1e6
except Exception as e:
    print('memory_stats unavailable:', e)
steady = timer.times[1:] if len(timer.times) > 1 else timer.times  # drop epoch 0 (compile)
epoch_s = float(np.median(steady))
print(f'median steady epoch: {epoch_s:.1f}s   peak_mem: {peak_mb} MB')

## 5. Sample outputs AFTER fine-tuning

In [ ]:
after = sample(gemma, 'after')

## 6. Record results
Append one row per device to `gemma_lora_results.csv` and download it.

In [ ]:
row = dict(**ENV, seqlen=SEQLEN, epochs=EPOCHS, batch=BATCH, lora_rank=LORA_RANK,
           load_s=round(load_s, 1), train_total_s=round(train_s, 1),
           epoch_s=round(epoch_s, 2), peak_mem_mb=(round(peak_mb, 1) if peak_mb else None),
           epoch_times=json.dumps([round(t, 1) for t in timer.times]))
path = 'gemma_lora_results.csv'
pd.DataFrame([row]).to_csv(path, mode='a', header=not os.path.exists(path), index=False)

# Save the before/after generations too.
with open(f'gemma_samples_{RUN_ID}.txt', 'w') as f:
    for p, b, a in zip(PROMPTS, before, after):
        f.write(f'PROMPT: {p}\n--- BEFORE ---\n{b}\n--- AFTER ---\n{a}\n\n{"="*60}\n')
print('wrote', path, 'and samples')
pd.read_csv(path)

In [ ]:
try:
    from google.colab import files
    files.download('gemma_lora_results.csv')
    files.download(f'gemma_samples_{RUN_ID}.txt')
except Exception as e:
    print('not on Colab; files are in', os.getcwd())

## 7. For the report
1. **Time per epoch** on v5e-1 vs T4, and peak memory on each.
2. Two or three generated responses **before vs after** fine-tuning (from the samples file).
3. Is the TPU-vs-GPU speed gap here similar to the CNN gap in Part 2? Explain why or why not
   (hint: this is a 1B-parameter transformer dominated by large matmuls, bf16 on the MXU,
   vs. a tiny CNN; relate to arithmetic intensity / roofline as in Wang et al. 2019).
4. If you reduced `SEQLEN` for the T4, say so — the same value was used on both devices.